In [1]:
# Cell 1: SPI-3 investigation setup and CHIRPS raster metadata inspection

import sys
from pathlib import Path

import pandas as pd
import rasterio

# tests/spi3_investigate.ipynb → RozviDrought project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "tests":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

CHIRPS_PATH = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model\precipitation"
) / "zimbabwe_chirps_monthly_1981_2024.tif"

if not CHIRPS_PATH.exists():
    raise FileNotFoundError(f"CHIRPS raster not found: {CHIRPS_PATH}")

with rasterio.open(CHIRPS_PATH) as src:
    print("CHIRPS raster:", CHIRPS_PATH)
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Band count:", src.count)
    print("Data type:", src.dtypes[0])
    print("NoData:", src.nodata)
    print("Bounds:", src.bounds)
    print("Transform:", src.transform)

    band_descriptions = list(src.descriptions)

print("\nFirst 12 band descriptions:")
print(band_descriptions[:12])

print("\nLast 12 band descriptions:")
print(band_descriptions[-12:])

expected_months = (2024 - 1981 + 1) * 12
print("\nExpected monthly bands for 1981-2024:", expected_months)

CHIRPS raster: C:\Projects\ramangwana\risks\data\drought_model\precipitation\zimbabwe_chirps_monthly_1981_2024.tif
CRS: EPSG:4326
Width: 175
Height: 153
Band count: 3168
Data type: float32
NoData: None
Bounds: BoundingBox(left=25.197743719552577, bottom=-22.457882102988037, right=33.05800245559839, top=-15.585770179473698)
Transform: | 0.04, 0.00, 25.20|
| 0.00,-0.04,-15.59|
| 0.00, 0.00, 1.00|

First 12 band descriptions:
['19810101_precipitation', '19810106_precipitation', '19810111_precipitation', '19810116_precipitation', '19810121_precipitation', '19810126_precipitation', '19810201_precipitation', '19810206_precipitation', '19810211_precipitation', '19810216_precipitation', '19810221_precipitation', '19810226_precipitation']

Last 12 band descriptions:
['20241101_precipitation', '20241106_precipitation', '20241111_precipitation', '20241116_precipitation', '20241121_precipitation', '20241126_precipitation', '20241201_precipitation', '20241206_precipitation', '20241211_precipitation

In [2]:
# Cell 2: Build CHIRPS band index and verify monthly grouping

band_index_rows = []

for band_number, desc in enumerate(band_descriptions, start=1):
    date_text = desc.split("_")[0]
    date = pd.to_datetime(date_text, format="%Y%m%d")

    band_index_rows.append({
        "band_number": band_number,
        "description": desc,
        "date": date,
        "yyyymm": date.strftime("%Y%m"),
        "year": date.year,
        "month": date.month,
        "day": date.day,
    })

chirps_band_index_df = pd.DataFrame(band_index_rows)

monthly_band_counts_df = (
    chirps_band_index_df
    .groupby("yyyymm")
    .size()
    .reset_index(name="band_count")
)

print("Band index rows:", len(chirps_band_index_df))
print("Unique months:", chirps_band_index_df["yyyymm"].nunique())
print("Month range:", chirps_band_index_df["yyyymm"].min(), "→", chirps_band_index_df["yyyymm"].max())

print("\nMonthly band count summary:")
print(monthly_band_counts_df["band_count"].value_counts().sort_index())

print("\nFirst 12 bands:")
print(chirps_band_index_df.head(12))

print("\nLast 12 bands:")
print(chirps_band_index_df.tail(12))

Band index rows: 3168
Unique months: 528
Month range: 198101 → 202412

Monthly band count summary:
band_count
6    528
Name: count, dtype: int64

First 12 bands:
    band_number             description       date  yyyymm  year  month  day
0             1  19810101_precipitation 1981-01-01  198101  1981      1    1
1             2  19810106_precipitation 1981-01-06  198101  1981      1    6
2             3  19810111_precipitation 1981-01-11  198101  1981      1   11
3             4  19810116_precipitation 1981-01-16  198101  1981      1   16
4             5  19810121_precipitation 1981-01-21  198101  1981      1   21
5             6  19810126_precipitation 1981-01-26  198101  1981      1   26
6             7  19810201_precipitation 1981-02-01  198102  1981      2    1
7             8  19810206_precipitation 1981-02-06  198102  1981      2    6
8             9  19810211_precipitation 1981-02-11  198102  1981      2   11
9            10  19810216_precipitation 1981-02-16  198102  1981    

In [3]:
# Cell 3: Convert 6-step CHIRPS bands into monthly rainfall totals

import gc
import numpy as np

monthly_yyyymm = monthly_band_counts_df["yyyymm"].tolist()
monthly_precip_arrays = []

print("Building monthly precipitation totals...")
print("Total months:", len(monthly_yyyymm))

with rasterio.open(CHIRPS_PATH) as src:
    raster_profile = src.profile.copy()
    raster_transform = src.transform
    raster_crs = src.crs
    raster_bounds = src.bounds
    raster_height = src.height
    raster_width = src.width

    for i, yyyymm in enumerate(monthly_yyyymm, start=1):
        month_bands = chirps_band_index_df.loc[
            chirps_band_index_df["yyyymm"].eq(yyyymm),
            "band_number",
        ].tolist()

        if len(month_bands) != 6:
            raise ValueError(f"{yyyymm} has {len(month_bands)} bands, expected 6.")

        month_stack = src.read(month_bands).astype("float32")
        month_total = np.nansum(month_stack, axis=0).astype("float32")

        monthly_precip_arrays.append(month_total)

        if i == 1 or i % 60 == 0 or i == len(monthly_yyyymm):
            print(f"[{i}/{len(monthly_yyyymm)}] processed {yyyymm}")

        del month_stack
        gc.collect()

monthly_precip_stack = np.stack(monthly_precip_arrays, axis=0).astype("float32")

del monthly_precip_arrays
gc.collect()

print("\nMonthly precipitation stack built.")
print("Shape:", monthly_precip_stack.shape)
print("Expected shape:", (len(monthly_yyyymm), raster_height, raster_width))
print("Month range:", monthly_yyyymm[0], "→", monthly_yyyymm[-1])
print("Min rainfall:", float(np.nanmin(monthly_precip_stack)))
print("Mean rainfall:", float(np.nanmean(monthly_precip_stack)))
print("Max rainfall:", float(np.nanmax(monthly_precip_stack)))

Building monthly precipitation totals...
Total months: 528
[1/528] processed 198101
[60/528] processed 198512
[120/528] processed 199012
[180/528] processed 199512
[240/528] processed 200012
[300/528] processed 200512
[360/528] processed 201012
[420/528] processed 201512
[480/528] processed 202012
[528/528] processed 202412

Monthly precipitation stack built.
Shape: (528, 153, 175)
Expected shape: (528, 153, 175)
Month range: 198101 → 202412
Min rainfall: 3.0153799457366404e-09
Mean rainfall: 50.542381286621094
Max rainfall: 1518.7164306640625


In [4]:
# Cell 4: Compute rolling 3-month rainfall totals for SPI-3

spi3_window_totals = np.full_like(monthly_precip_stack, np.nan, dtype="float32")

print("Computing rolling 3-month rainfall totals...")

for i in range(2, monthly_precip_stack.shape[0]):
    spi3_window_totals[i] = (
        monthly_precip_stack[i - 2]
        + monthly_precip_stack[i - 1]
        + monthly_precip_stack[i]
    )

    if i == 2 or (i + 1) % 60 == 0 or i == monthly_precip_stack.shape[0] - 1:
        print(f"[{i + 1}/{monthly_precip_stack.shape[0]}] processed {monthly_yyyymm[i]}")

print("\nSPI-3 rainfall-window totals built.")
print("Shape:", spi3_window_totals.shape)
print("First valid SPI-3 month:", monthly_yyyymm[2])
print("Last SPI-3 month:", monthly_yyyymm[-1])
print("Min 3-month rainfall:", float(np.nanmin(spi3_window_totals)))
print("Mean 3-month rainfall:", float(np.nanmean(spi3_window_totals)))
print("Max 3-month rainfall:", float(np.nanmax(spi3_window_totals)))

Computing rolling 3-month rainfall totals...
[3/528] processed 198103
[60/528] processed 198512
[120/528] processed 199012
[180/528] processed 199512
[240/528] processed 200012
[300/528] processed 200512
[360/528] processed 201012
[420/528] processed 201512
[480/528] processed 202012
[528/528] processed 202412

SPI-3 rainfall-window totals built.
Shape: (528, 153, 175)
First valid SPI-3 month: 198103
Last SPI-3 month: 202412
Min 3-month rainfall: 0.09191208332777023
Mean 3-month rainfall: 150.59365844726562
Max 3-month rainfall: 3112.725341796875


In [5]:
# Cell 5: Compute empirical SPI-3 from rolling 3-month rainfall totals

from scipy.stats import norm, rankdata

monthly_dates = pd.to_datetime(monthly_yyyymm, format="%Y%m")
monthly_month_numbers = np.array([d.month for d in monthly_dates])

spi3_stack = np.full_like(spi3_window_totals, np.nan, dtype="float32")

print("Computing empirical SPI-3...")
print("Method: rank-based empirical probability → standard normal score")
print("Calendar months processed separately.")

for calendar_month in range(1, 13):
    month_positions = np.where(monthly_month_numbers == calendar_month)[0]

    # Drop all-NaN early windows, especially Jan/Feb 1981.
    valid_positions = [
        pos for pos in month_positions
        if np.isfinite(spi3_window_totals[pos]).any()
    ]

    month_values = spi3_window_totals[valid_positions].astype("float32")

    ranks = rankdata(month_values, axis=0, method="average")

    n = month_values.shape[0]

    # Gringorten plotting position, commonly used for precipitation frequency work.
    probs = (ranks - 0.44) / (n + 0.12)
    probs = np.clip(probs, 1e-6, 1 - 1e-6)

    spi_values = norm.ppf(probs).astype("float32")

    spi3_stack[valid_positions] = spi_values

    print(
        f"Month {calendar_month:02d} | "
        f"samples={n} | "
        f"positions={valid_positions[0]}-{valid_positions[-1]}"
    )

print("\nSPI-3 stack built.")
print("Shape:", spi3_stack.shape)
print("Min SPI-3:", float(np.nanmin(spi3_stack)))
print("Mean SPI-3:", float(np.nanmean(spi3_stack)))
print("Max SPI-3:", float(np.nanmax(spi3_stack)))

Computing empirical SPI-3...
Method: rank-based empirical probability → standard normal score
Calendar months processed separately.
Month 01 | samples=43 | positions=12-516
Month 02 | samples=43 | positions=13-517
Month 03 | samples=44 | positions=2-518
Month 04 | samples=44 | positions=3-519
Month 05 | samples=44 | positions=4-520
Month 06 | samples=44 | positions=5-521
Month 07 | samples=44 | positions=6-522
Month 08 | samples=44 | positions=7-523
Month 09 | samples=44 | positions=8-524
Month 10 | samples=44 | positions=9-525
Month 11 | samples=44 | positions=10-526
Month 12 | samples=44 | positions=11-527

SPI-3 stack built.
Shape: (528, 153, 175)
Min SPI-3: -2.235488176345825
Mean SPI-3: -6.967298986637616e-07
Max SPI-3: 2.235488176345825


In [6]:
# Cell 6: Classify SPI-3 values into drought classes

spi3_class_stack = np.full(spi3_stack.shape, -1, dtype="int8")

# Class codes:
# -1 = missing
#  0 = normal / no drought
#  1 = moderate drought
#  2 = severe drought
#  3 = extreme drought

valid = np.isfinite(spi3_stack)

spi3_class_stack[valid & (spi3_stack > -1.0)] = 0
spi3_class_stack[valid & (spi3_stack <= -1.0) & (spi3_stack > -1.5)] = 1
spi3_class_stack[valid & (spi3_stack <= -1.5) & (spi3_stack > -2.0)] = 2
spi3_class_stack[valid & (spi3_stack <= -2.0)] = 3

class_summary = []

for class_code, class_name in {
    -1: "missing",
    0: "normal",
    1: "moderate",
    2: "severe",
    3: "extreme",
}.items():
    count = int((spi3_class_stack == class_code).sum())
    class_summary.append({
        "class_code": class_code,
        "class_name": class_name,
        "pixel_month_count": count,
    })

spi3_class_summary_df = pd.DataFrame(class_summary)

print("SPI-3 class stack built.")
print("Shape:", spi3_class_stack.shape)
print("\nClass summary:")
print(spi3_class_summary_df)

SPI-3 class stack built.
Shape: (528, 153, 175)

Class summary:
   class_code class_name  pixel_month_count
0          -1    missing              53550
1           0     normal           11834593
2           1   moderate            1285198
3           2     severe             642566
4           3    extreme             321293


In [7]:
# Cell 7: Save SPI-3 rasters and metadata to data/backtests/spi3

SPI3_OUTPUT_DIR = PROJECT_ROOT.parent / "data" / "backtests" / "spi3"
SPI3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPI3_VALUE_TIF = SPI3_OUTPUT_DIR / "zimbabwe_chirps_spi3_empirical_198103_202412.tif"
SPI3_CLASS_TIF = SPI3_OUTPUT_DIR / "zimbabwe_chirps_spi3_class_198103_202412.tif"
SPI3_MONTH_INDEX_CSV = SPI3_OUTPUT_DIR / "zimbabwe_chirps_spi3_month_index.csv"

valid_spi3_positions = [
    i for i, yyyymm in enumerate(monthly_yyyymm)
    if yyyymm >= "198103"
]

valid_spi3_months = [monthly_yyyymm[i] for i in valid_spi3_positions]

spi3_value_profile = raster_profile.copy()
spi3_value_profile.update(
    count=len(valid_spi3_positions),
    dtype="float32",
    nodata=-9999.0,
    compress="deflate",
)

spi3_class_profile = raster_profile.copy()
spi3_class_profile.update(
    count=len(valid_spi3_positions),
    dtype="int8",
    nodata=-1,
    compress="deflate",
)

print("Saving SPI-3 value raster...")
with rasterio.open(SPI3_VALUE_TIF, "w", **spi3_value_profile) as dst:
    for out_band, src_pos in enumerate(valid_spi3_positions, start=1):
        arr = spi3_stack[src_pos].astype("float32")
        arr = np.where(np.isfinite(arr), arr, -9999.0).astype("float32")
        dst.write(arr, out_band)
        dst.set_band_description(out_band, f"{monthly_yyyymm[src_pos]}_spi3")

        if out_band == 1 or out_band % 60 == 0 or out_band == len(valid_spi3_positions):
            print(f"[{out_band}/{len(valid_spi3_positions)}] saved SPI-3 {monthly_yyyymm[src_pos]}")

print("\nSaving SPI-3 class raster...")
with rasterio.open(SPI3_CLASS_TIF, "w", **spi3_class_profile) as dst:
    for out_band, src_pos in enumerate(valid_spi3_positions, start=1):
        dst.write(spi3_class_stack[src_pos].astype("int8"), out_band)
        dst.set_band_description(out_band, f"{monthly_yyyymm[src_pos]}_spi3_class")

        if out_band == 1 or out_band % 60 == 0 or out_band == len(valid_spi3_positions):
            print(f"[{out_band}/{len(valid_spi3_positions)}] saved SPI-3 class {monthly_yyyymm[src_pos]}")

spi3_month_index_df = pd.DataFrame({
    "band_number": range(1, len(valid_spi3_months) + 1),
    "yyyymm": valid_spi3_months,
})

spi3_month_index_df.to_csv(SPI3_MONTH_INDEX_CSV, index=False)

print("\nSaved SPI-3 outputs:")
print("SPI-3 values:", SPI3_VALUE_TIF)
print("SPI-3 classes:", SPI3_CLASS_TIF)
print("Month index:", SPI3_MONTH_INDEX_CSV)
print("Bands saved:", len(valid_spi3_months))

Saving SPI-3 value raster...
[1/526] saved SPI-3 198103
[60/526] saved SPI-3 198602
[120/526] saved SPI-3 199102
[180/526] saved SPI-3 199602
[240/526] saved SPI-3 200102
[300/526] saved SPI-3 200602
[360/526] saved SPI-3 201102
[420/526] saved SPI-3 201602
[480/526] saved SPI-3 202102
[526/526] saved SPI-3 202412

Saving SPI-3 class raster...
[1/526] saved SPI-3 class 198103
[60/526] saved SPI-3 class 198602
[120/526] saved SPI-3 class 199102
[180/526] saved SPI-3 class 199602
[240/526] saved SPI-3 class 200102
[300/526] saved SPI-3 class 200602
[360/526] saved SPI-3 class 201102
[420/526] saved SPI-3 class 201602
[480/526] saved SPI-3 class 202102
[526/526] saved SPI-3 class 202412

Saved SPI-3 outputs:
SPI-3 values: C:\Projects\Infer RozviDrought\data\backtests\spi3\zimbabwe_chirps_spi3_empirical_198103_202412.tif
SPI-3 classes: C:\Projects\Infer RozviDrought\data\backtests\spi3\zimbabwe_chirps_spi3_class_198103_202412.tif
Month index: C:\Projects\Infer RozviDrought\data\backtests\s

In [8]:
# Cell 8: Load saved SPI-3 outputs and controlled ADM2 backtest predictions

SPI3_OUTPUT_DIR = PROJECT_ROOT.parent / "data" / "backtests" / "spi3"

SPI3_VALUE_TIF = SPI3_OUTPUT_DIR / "zimbabwe_chirps_spi3_empirical_198103_202412.tif"
SPI3_CLASS_TIF = SPI3_OUTPUT_DIR / "zimbabwe_chirps_spi3_class_198103_202412.tif"
SPI3_MONTH_INDEX_CSV = SPI3_OUTPUT_DIR / "zimbabwe_chirps_spi3_month_index.csv"

CONTROLLED_BATCH_PARQUET = (
    PROJECT_ROOT.parent
    / "data"
    / "backtests"
    / "controlled_adm2_backtest_2023_2024.parquet"
)

CONTROLLED_BATCH_CSV = (
    PROJECT_ROOT.parent
    / "data"
    / "backtests"
    / "controlled_adm2_backtest_2023_2024.csv"
)

for path in [SPI3_VALUE_TIF, SPI3_CLASS_TIF, SPI3_MONTH_INDEX_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"Missing SPI-3 output: {path}")

if CONTROLLED_BATCH_PARQUET.exists():
    controlled_batch_df = pd.read_parquet(CONTROLLED_BATCH_PARQUET)
elif CONTROLLED_BATCH_CSV.exists():
    controlled_batch_df = pd.read_csv(CONTROLLED_BATCH_CSV)
else:
    raise FileNotFoundError(
        "Controlled batch output not found. Save controlled_batch_df first from backtests.ipynb."
    )

spi3_month_index_df = pd.read_csv(SPI3_MONTH_INDEX_CSV)
spi3_month_index_df["yyyymm"] = spi3_month_index_df["yyyymm"].astype(str)

controlled_batch_df["yyyymm"] = controlled_batch_df["yyyymm"].astype(str)

print("Loaded controlled backtest predictions:", len(controlled_batch_df))
print("Prediction months:", controlled_batch_df["yyyymm"].drop_duplicates().tolist())

print("\nLoaded SPI-3 month index:", len(spi3_month_index_df))
print("SPI-3 month range:", spi3_month_index_df["yyyymm"].min(), "→", spi3_month_index_df["yyyymm"].max())

with rasterio.open(SPI3_CLASS_TIF) as src:
    print("\nSPI-3 class raster:")
    print("Path:", SPI3_CLASS_TIF)
    print("Bands:", src.count)
    print("CRS:", src.crs)
    print("Shape:", (src.height, src.width))
    print("NoData:", src.nodata)

Loaded controlled backtest predictions: 35
Prediction months: ['202310', '202311', '202312', '202401', '202402', '202403', '202404']

Loaded SPI-3 month index: 526
SPI-3 month range: 198103 → 202412

SPI-3 class raster:
Path: C:\Projects\Infer RozviDrought\data\backtests\spi3\zimbabwe_chirps_spi3_class_198103_202412.tif
Bands: 526
CRS: EPSG:4326
Shape: (153, 175)
NoData: -1.0


In [11]:
# Cell 9 fix: define model class mapping, then rerun SPI-3 comparison

CLASS_TO_SEVERITY = {
    "normal": 0,
    "moderate": 2,
    "severe": 3,
    "extreme": 4,
}

def model_class_to_severity(model_class):
    if model_class is None:
        return None

    return CLASS_TO_SEVERITY.get(str(model_class).lower())


SPI3_CLASS_NAMES = {
    -1: "missing",
    0: "normal",
    1: "moderate",
    2: "severe",
    3: "extreme",
}

SPI3_CLASS_TO_SEVERITY = {
    -1: None,
    0: 0,
    1: 2,
    2: 3,
    3: 4,
}

def observed_spi3_comparison(model_class, observed_class_code):
    observed_class_code = int(observed_class_code)

    observed_class = SPI3_CLASS_NAMES.get(observed_class_code, "unknown")
    observed_severity = SPI3_CLASS_TO_SEVERITY.get(observed_class_code)
    model_severity = model_class_to_severity(model_class)

    if observed_severity is None or model_severity is None:
        return {
            "observed_spi3_class": observed_class,
            "observed_spi3_severity": observed_severity,
            "model_spi3_severity": model_severity,
            "spi3_severity_gap": None,
            "spi3_comparison": "missing",
        }

    gap = int(model_severity) - int(observed_severity)

    if gap == 0:
        comparison = "matched"
    elif gap < 0:
        comparison = "underestimated"
    else:
        comparison = "overestimated"

    return {
        "observed_spi3_class": observed_class,
        "observed_spi3_severity": observed_severity,
        "model_spi3_severity": model_severity,
        "spi3_severity_gap": gap,
        "spi3_comparison": comparison,
    }


spi3_lookup_rows = []

print("Comparing model predictions to observed SPI-3...")
print("Rows to process:", len(controlled_batch_df))

with rasterio.open(SPI3_CLASS_TIF) as src:
    for i, (_, row) in enumerate(controlled_batch_df.iterrows(), start=1):
        yyyymm = str(row["yyyymm"])

        band_match = spi3_month_index_df[spi3_month_index_df["yyyymm"].eq(yyyymm)]
        if band_match.empty:
            raise ValueError(f"No SPI-3 band found for {yyyymm}")

        band_number = int(band_match.iloc[0]["band_number"])
        spi3_class_arr = src.read(band_number)

        admin_match = usable_admin_df[
            usable_admin_df["adm2_code"].astype(int).eq(int(row["adm2_code"]))
        ]

        if admin_match.empty:
            raise ValueError(f"No ADM2 geometry found for code {row['adm2_code']}")

        geom = admin_match.iloc[0]["polygon_geometry"]

        mask = rasterio.features.geometry_mask(
            [geom.__geo_interface__],
            out_shape=spi3_class_arr.shape,
            transform=src.transform,
            invert=True,
        )

        values = spi3_class_arr[mask]
        values = values[values != src.nodata]

        if len(values) == 0:
            observed_class_code = -1
            observed_pixel_count = 0
        else:
            unique_values, counts = np.unique(values.astype("int16"), return_counts=True)
            observed_class_code = int(unique_values[np.argmax(counts)])
            observed_pixel_count = int(len(values))

        comparison = observed_spi3_comparison(
            model_class=row["dominant_class"],
            observed_class_code=observed_class_code,
        )

        spi3_lookup_rows.append({
            **row.to_dict(),
            "spi3_band_number": band_number,
            "observed_spi3_class_code": observed_class_code,
            "observed_spi3_pixel_count": observed_pixel_count,
            **comparison,
        })

        if i == 1 or i % 5 == 0 or i == len(controlled_batch_df):
            print(f"[{i}/{len(controlled_batch_df)}] processed {row['adm2_name']} | {yyyymm}")

spi3_comparison_df = pd.DataFrame(spi3_lookup_rows)

print("\nSPI-3 comparison rows:", len(spi3_comparison_df))
print(
    spi3_comparison_df[
        [
            "adm1_name",
            "adm2_name",
            "yyyymm",
            "dominant_class",
            "observed_spi3_class",
            "spi3_comparison",
            "spi3_severity_gap",
            "observed_spi3_pixel_count",
        ]
    ]
)

print("\nSPI-3 comparison summary:")
print(
    spi3_comparison_df
    .groupby(["spi3_comparison"], dropna=False)
    .size()
    .reset_index(name="rows")
)

Comparing model predictions to observed SPI-3...
Rows to process: 35
[1/35] processed Chivi | 202310
[5/35] processed Chivi | 202402
[10/35] processed Buhera | 202312
[15/35] processed Gwanda | 202310
[20/35] processed Gwanda | 202403
[25/35] processed Beitbridge | 202401
[30/35] processed Mwenezi | 202311
[35/35] processed Mwenezi | 202404

SPI-3 comparison rows: 35
             adm1_name   adm2_name  yyyymm dominant_class observed_spi3_class  \
0             Masvingo       Chivi  202310         severe              normal   
1             Masvingo       Chivi  202311         normal              normal   
2             Masvingo       Chivi  202312         normal              normal   
3             Masvingo       Chivi  202401         normal              normal   
4             Masvingo       Chivi  202402         normal              normal   
5             Masvingo       Chivi  202403         normal              severe   
6             Masvingo       Chivi  202404         normal      

In [12]:
# Cell 10: Save SPI-3 comparison outputs and summarize accuracy by month/district

SPI3_COMPARE_OUTPUT_DIR = PROJECT_ROOT.parent / "data" / "backtests" / "spi3_comparison"
SPI3_COMPARE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPI3_COMPARISON_CSV = SPI3_COMPARE_OUTPUT_DIR / "adm2_model_vs_observed_spi3_2023_2024.csv"
SPI3_COMPARISON_PARQUET = SPI3_COMPARE_OUTPUT_DIR / "adm2_model_vs_observed_spi3_2023_2024.parquet"
SPI3_MONTH_SUMMARY_CSV = SPI3_COMPARE_OUTPUT_DIR / "spi3_accuracy_by_month_2023_2024.csv"
SPI3_DISTRICT_SUMMARY_CSV = SPI3_COMPARE_OUTPUT_DIR / "spi3_accuracy_by_district_2023_2024.csv"

spi3_comparison_df.to_csv(SPI3_COMPARISON_CSV, index=False)
spi3_comparison_df.to_parquet(SPI3_COMPARISON_PARQUET, index=False)

month_summary_df = (
    spi3_comparison_df
    .groupby(["yyyymm", "spi3_comparison"], dropna=False)
    .size()
    .reset_index(name="rows")
    .pivot(index="yyyymm", columns="spi3_comparison", values="rows")
    .fillna(0)
    .reset_index()
)

for col in ["matched", "underestimated", "overestimated", "missing"]:
    if col not in month_summary_df.columns:
        month_summary_df[col] = 0

month_summary_df["total"] = (
    month_summary_df["matched"]
    + month_summary_df["underestimated"]
    + month_summary_df["overestimated"]
    + month_summary_df["missing"]
)

month_summary_df["match_rate"] = month_summary_df["matched"] / month_summary_df["total"]

district_summary_df = (
    spi3_comparison_df
    .groupby(["adm1_name", "adm2_name", "spi3_comparison"], dropna=False)
    .size()
    .reset_index(name="rows")
    .pivot(index=["adm1_name", "adm2_name"], columns="spi3_comparison", values="rows")
    .fillna(0)
    .reset_index()
)

for col in ["matched", "underestimated", "overestimated", "missing"]:
    if col not in district_summary_df.columns:
        district_summary_df[col] = 0

district_summary_df["total"] = (
    district_summary_df["matched"]
    + district_summary_df["underestimated"]
    + district_summary_df["overestimated"]
    + district_summary_df["missing"]
)

district_summary_df["match_rate"] = district_summary_df["matched"] / district_summary_df["total"]

month_summary_df.to_csv(SPI3_MONTH_SUMMARY_CSV, index=False)
district_summary_df.to_csv(SPI3_DISTRICT_SUMMARY_CSV, index=False)

print("Saved SPI-3 comparison outputs:")
print("Comparison CSV:", SPI3_COMPARISON_CSV)
print("Comparison Parquet:", SPI3_COMPARISON_PARQUET)
print("Month summary CSV:", SPI3_MONTH_SUMMARY_CSV)
print("District summary CSV:", SPI3_DISTRICT_SUMMARY_CSV)

print("\nOverall SPI-3 comparison:")
print(
    spi3_comparison_df
    .groupby("spi3_comparison")
    .size()
    .reset_index(name="rows")
)

print("\nAccuracy by month:")
print(month_summary_df[["yyyymm", "matched", "underestimated", "overestimated", "total", "match_rate"]])

print("\nAccuracy by district:")
print(district_summary_df[["adm1_name", "adm2_name", "matched", "underestimated", "overestimated", "total", "match_rate"]])

Saved SPI-3 comparison outputs:
Comparison CSV: C:\Projects\Infer RozviDrought\data\backtests\spi3_comparison\adm2_model_vs_observed_spi3_2023_2024.csv
Comparison Parquet: C:\Projects\Infer RozviDrought\data\backtests\spi3_comparison\adm2_model_vs_observed_spi3_2023_2024.parquet
Month summary CSV: C:\Projects\Infer RozviDrought\data\backtests\spi3_comparison\spi3_accuracy_by_month_2023_2024.csv
District summary CSV: C:\Projects\Infer RozviDrought\data\backtests\spi3_comparison\spi3_accuracy_by_district_2023_2024.csv

Overall SPI-3 comparison:
  spi3_comparison  rows
0         matched    19
1   overestimated     4
2  underestimated    12

Accuracy by month:
spi3_comparison  yyyymm  matched  underestimated  overestimated  total  \
0                202310      1.0             0.0            4.0    5.0   
1                202311      3.0             2.0            0.0    5.0   
2                202312      5.0             0.0            0.0    5.0   
3                202401      5.0       

In [13]:
# Cell 11: Compute calendar-month Gamma SPI-3 for target comparison months only

from scipy.stats import gamma, norm
import numpy as np
import pandas as pd
import gc

TARGET_YYYYMM = sorted(controlled_batch_df["yyyymm"].astype(str).unique().tolist())

monthly_dates = pd.to_datetime(monthly_yyyymm, format="%Y%m")
monthly_month_numbers = np.array([d.month for d in monthly_dates])
monthly_yyyymm_array = np.array(monthly_yyyymm)

gamma_spi3_target_stack = {}
gamma_spi3_class_target_stack = {}

print("Computing calendar-month Gamma SPI-3 for target months only.")
print("Target months:", TARGET_YYYYMM)
print("Method: calendar-month Gamma distribution, vectorized method-of-moments fit")

for target_yyyymm in TARGET_YYYYMM:
    target_positions = np.where(monthly_yyyymm_array == target_yyyymm)[0]

    if len(target_positions) != 1:
        raise ValueError(f"Could not find unique target month: {target_yyyymm}")

    target_pos = int(target_positions[0])
    calendar_month = int(monthly_month_numbers[target_pos])

    climatology_positions = np.where(monthly_month_numbers == calendar_month)[0]
    climatology_positions = [
        pos for pos in climatology_positions
        if np.isfinite(spi3_window_totals[pos]).any()
    ]

    clim = spi3_window_totals[climatology_positions].astype("float32")
    target_values = spi3_window_totals[target_pos].astype("float32")

    positive_clim = np.where(clim > 0, clim, np.nan)

    mean = np.nanmean(positive_clim, axis=0)
    var = np.nanvar(positive_clim, axis=0)

    valid_fit = np.isfinite(mean) & np.isfinite(var) & (mean > 0) & (var > 0)

    shape_param = np.full(mean.shape, np.nan, dtype="float32")
    scale_param = np.full(mean.shape, np.nan, dtype="float32")

    shape_param[valid_fit] = (mean[valid_fit] ** 2) / var[valid_fit]
    scale_param[valid_fit] = var[valid_fit] / mean[valid_fit]

    prob_zero = np.mean(clim <= 0, axis=0).astype("float32")

    cdf = np.full(target_values.shape, np.nan, dtype="float32")

    positive_target = target_values > 0
    valid_target = valid_fit & positive_target

    cdf[valid_target] = (
        prob_zero[valid_target]
        + (1 - prob_zero[valid_target])
        * gamma.cdf(
            target_values[valid_target],
            a=shape_param[valid_target],
            scale=scale_param[valid_target],
        )
    )

    zero_target = valid_fit & ~positive_target
    cdf[zero_target] = prob_zero[zero_target]

    cdf = np.clip(cdf, 0.0001, 0.9999)

    gamma_spi3 = norm.ppf(cdf).astype("float32")

    gamma_spi3_class = np.full(gamma_spi3.shape, -1, dtype="int8")
    valid = np.isfinite(gamma_spi3)

    gamma_spi3_class[valid & (gamma_spi3 > -1.0)] = 0
    gamma_spi3_class[valid & (gamma_spi3 <= -1.0) & (gamma_spi3 > -1.5)] = 1
    gamma_spi3_class[valid & (gamma_spi3 <= -1.5) & (gamma_spi3 > -2.0)] = 2
    gamma_spi3_class[valid & (gamma_spi3 <= -2.0)] = 3

    gamma_spi3_target_stack[target_yyyymm] = gamma_spi3
    gamma_spi3_class_target_stack[target_yyyymm] = gamma_spi3_class

    unique_classes, counts = np.unique(gamma_spi3_class, return_counts=True)
    class_counts = dict(zip(unique_classes.tolist(), counts.tolist()))

    print(
        f"{target_yyyymm} | calendar_month={calendar_month:02d} | "
        f"samples={len(climatology_positions)} | "
        f"min={float(np.nanmin(gamma_spi3)):.3f} | "
        f"mean={float(np.nanmean(gamma_spi3)):.3f} | "
        f"max={float(np.nanmax(gamma_spi3)):.3f} | "
        f"classes={class_counts}"
    )

    del clim, positive_clim, mean, var, shape_param, scale_param, cdf
    gc.collect()

print("\nGamma SPI-3 target stacks ready.")
print("Months computed:", list(gamma_spi3_target_stack.keys()))

Computing calendar-month Gamma SPI-3 for target months only.
Target months: ['202310', '202311', '202312', '202401', '202402', '202403', '202404']
Method: calendar-month Gamma distribution, vectorized method-of-moments fit
202310 | calendar_month=10 | samples=44 | min=-1.120 | mean=0.776 | max=3.646 | classes={0: 26737, 1: 38}
202311 | calendar_month=11 | samples=44 | min=-3.628 | mean=-1.260 | max=2.432 | classes={0: 11605, 1: 2799, 2: 2321, 3: 10050}
202312 | calendar_month=12 | samples=44 | min=-2.288 | mean=-0.035 | max=1.988 | classes={0: 21636, 1: 3823, 2: 1214, 3: 102}
202401 | calendar_month=01 | samples=43 | min=-1.517 | mean=-0.334 | max=0.994 | classes={0: 25375, 1: 1399, 2: 1}
202402 | calendar_month=02 | samples=43 | min=-2.095 | mean=-0.744 | max=1.192 | classes={0: 17258, 1: 7424, 2: 2056, 3: 37}
202403 | calendar_month=03 | samples=44 | min=-3.494 | mean=-1.653 | max=-0.409 | classes={0: 3389, 1: 7593, 2: 8912, 3: 6881}
202404 | calendar_month=04 | samples=44 | min=-2.9

In [14]:
# Cell 12: Compare saved model predictions against target-month Gamma SPI-3 classes

gamma_spi3_lookup_rows = []

print("Comparing model predictions to Gamma SPI-3 truth...")
print("Rows to process:", len(controlled_batch_df))

with rasterio.open(SPI3_CLASS_TIF) as src_ref:
    for i, (_, row) in enumerate(controlled_batch_df.iterrows(), start=1):
        yyyymm = str(row["yyyymm"])

        if yyyymm not in gamma_spi3_class_target_stack:
            raise ValueError(f"Gamma SPI-3 class not computed for {yyyymm}")

        gamma_class_arr = gamma_spi3_class_target_stack[yyyymm]

        admin_match = usable_admin_df[
            usable_admin_df["adm2_code"].astype(int).eq(int(row["adm2_code"]))
        ]

        if admin_match.empty:
            raise ValueError(f"No ADM2 geometry found for code {row['adm2_code']}")

        geom = admin_match.iloc[0]["polygon_geometry"]

        mask = rasterio.features.geometry_mask(
            [geom.__geo_interface__],
            out_shape=gamma_class_arr.shape,
            transform=src_ref.transform,
            invert=True,
        )

        values = gamma_class_arr[mask]
        values = values[values != -1]

        if len(values) == 0:
            observed_class_code = -1
            observed_pixel_count = 0
        else:
            unique_values, counts = np.unique(values.astype("int16"), return_counts=True)
            observed_class_code = int(unique_values[np.argmax(counts)])
            observed_pixel_count = int(len(values))

        comparison = observed_spi3_comparison(
            model_class=row["dominant_class"],
            observed_class_code=observed_class_code,
        )

        gamma_spi3_lookup_rows.append({
            **row.to_dict(),
            "gamma_observed_spi3_class_code": observed_class_code,
            "gamma_observed_spi3_pixel_count": observed_pixel_count,
            "gamma_observed_spi3_class": comparison["observed_spi3_class"],
            "gamma_observed_spi3_severity": comparison["observed_spi3_severity"],
            "gamma_spi3_severity_gap": comparison["spi3_severity_gap"],
            "gamma_spi3_comparison": comparison["spi3_comparison"],
        })

        if i == 1 or i % 5 == 0 or i == len(controlled_batch_df):
            print(f"[{i}/{len(controlled_batch_df)}] processed {row['adm2_name']} | {yyyymm}")

gamma_spi3_comparison_df = pd.DataFrame(gamma_spi3_lookup_rows)

print("\nGamma SPI-3 comparison rows:", len(gamma_spi3_comparison_df))
print(
    gamma_spi3_comparison_df[
        [
            "adm1_name",
            "adm2_name",
            "yyyymm",
            "dominant_class",
            "gamma_observed_spi3_class",
            "gamma_spi3_comparison",
            "gamma_spi3_severity_gap",
            "gamma_observed_spi3_pixel_count",
        ]
    ]
)

print("\nGamma SPI-3 comparison summary:")
print(
    gamma_spi3_comparison_df
    .groupby("gamma_spi3_comparison", dropna=False)
    .size()
    .reset_index(name="rows")
)

Comparing model predictions to Gamma SPI-3 truth...
Rows to process: 35
[1/35] processed Chivi | 202310
[5/35] processed Chivi | 202402
[10/35] processed Buhera | 202312
[15/35] processed Gwanda | 202310
[20/35] processed Gwanda | 202403
[25/35] processed Beitbridge | 202401
[30/35] processed Mwenezi | 202311
[35/35] processed Mwenezi | 202404

Gamma SPI-3 comparison rows: 35
             adm1_name   adm2_name  yyyymm dominant_class  \
0             Masvingo       Chivi  202310         severe   
1             Masvingo       Chivi  202311         normal   
2             Masvingo       Chivi  202312         normal   
3             Masvingo       Chivi  202401         normal   
4             Masvingo       Chivi  202402         normal   
5             Masvingo       Chivi  202403         normal   
6             Masvingo       Chivi  202404         normal   
7           Manicaland      Buhera  202310         severe   
8           Manicaland      Buhera  202311         normal   
9          

In [ ]:
# Cell 13: Save Gamma SPI-3 comparison outputs and pilot accuracy summaries

GAMMA_COMPARE_OUTPUT_DIR = PROJECT_ROOT.parent / "data" / "backtests" / "gamma_spi3_comparison"
GAMMA_COMPARE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GAMMA_COMPARISON_CSV = GAMMA_COMPARE_OUTPUT_DIR / "adm2_model_vs_gamma_spi3_2023_2024.csv"
GAMMA_COMPARISON_PARQUET = GAMMA_COMPARE_OUTPUT_DIR / "adm2_model_vs_gamma_spi3_2023_2024.parquet"
GAMMA_MONTH_SUMMARY_CSV = GAMMA_COMPARE_OUTPUT_DIR / "gamma_spi3_accuracy_by_month_2023_2024.csv"
GAMMA_DISTRICT_SUMMARY_CSV = GAMMA_COMPARE_OUTPUT_DIR / "gamma_spi3_accuracy_by_district_2023_2024.csv"

gamma_spi3_comparison_df.to_csv(GAMMA_COMPARISON_CSV, index=False)
gamma_spi3_comparison_df.to_parquet(GAMMA_COMPARISON_PARQUET, index=False)

gamma_month_summary_df = (
    gamma_spi3_comparison_df
    .groupby(["yyyymm", "gamma_spi3_comparison"], dropna=False)
    .size()
    .reset_index(name="rows")
    .pivot(index="yyyymm", columns="gamma_spi3_comparison", values="rows")
    .fillna(0)
    .reset_index()
)

for col in ["matched", "underestimated", "overestimated", "missing"]:
    if col not in gamma_month_summary_df.columns:
        gamma_month_summary_df[col] = 0

gamma_month_summary_df["total"] = (
    gamma_month_summary_df["matched"]
    + gamma_month_summary_df["underestimated"]
    + gamma_month_summary_df["overestimated"]
    + gamma_month_summary_df["missing"]
)

gamma_month_summary_df["match_rate"] = (
    gamma_month_summary_df["matched"] / gamma_month_summary_df["total"]
)

gamma_district_summary_df = (
    gamma_spi3_comparison_df
    .groupby(["adm1_name", "adm2_name", "gamma_spi3_comparison"], dropna=False)
    .size()
    .reset_index(name="rows")
    .pivot(index=["adm1_name", "adm2_name"], columns="gamma_spi3_comparison", values="rows")
    .fillna(0)
    .reset_index()
)

for col in ["matched", "underestimated", "overestimated", "missing"]:
    if col not in gamma_district_summary_df.columns:
        gamma_district_summary_df[col] = 0

gamma_district_summary_df["total"] = (
    gamma_district_summary_df["matched"]
    + gamma_district_summary_df["underestimated"]
    + gamma_district_summary_df["overestimated"]
    + gamma_district_summary_df["missing"]
)

gamma_district_summary_df["match_rate"] = (
    gamma_district_summary_df["matched"] / gamma_district_summary_df["total"]
)

gamma_month_summary_df.to_csv(GAMMA_MONTH_SUMMARY_CSV, index=False)
gamma_district_summary_df.to_csv(GAMMA_DISTRICT_SUMMARY_CSV, index=False)

overall_rows = len(gamma_spi3_comparison_df)
overall_matched = int((gamma_spi3_comparison_df["gamma_spi3_comparison"] == "matched").sum())
overall_accuracy = overall_matched / overall_rows if overall_rows else None

print("Saved Gamma SPI-3 comparison outputs:")
print("Comparison CSV:", GAMMA_COMPARISON_CSV)
print("Comparison Parquet:", GAMMA_COMPARISON_PARQUET)
print("Month summary CSV:", GAMMA_MONTH_SUMMARY_CSV)
print("District summary CSV:", GAMMA_DISTRICT_SUMMARY_CSV)

print("\nPilot Gamma SPI-3 accuracy:")
print("Matched rows:", overall_matched)
print("Total rows:", overall_rows)
print("Accuracy:", round(overall_accuracy, 4))

print("\nAccuracy by month:")
print(
    gamma_month_summary_df[
        ["yyyymm", "matched", "underestimated", "overestimated", "total", "match_rate"]
    ]
)

print("\nAccuracy by district:")
print(
    gamma_district_summary_df[
        ["adm1_name", "adm2_name", "matched", "underestimated", "overestimated", "total", "match_rate"]
    ]
)